# D-Fire — Phases 2/3: Training on Kaggle

Trains a YOLO detector using the project's config-driven pipeline, then
evaluates it (detection + image-level alert metrics + calibration).

**Before running:** enable GPU (Settings -> Accelerator -> GPU T4/P100) and
attach the D-Fire YOLO dataset. Set `DATA_ROOT` to its mount path.

Trains `configs/train_yolo_main.yaml` (YOLOv8m). Lower `epochs` in the
overrides below for a quick smoke test before the full run.

In [ ]:
# Environment setup
import sys, os
from pathlib import Path

# On Kaggle, clone or upload this repo, then point REPO_ROOT at it.
REPO_ROOT = Path.cwd()
while REPO_ROOT != REPO_ROOT.parent and not (REPO_ROOT / "src").is_dir():
    REPO_ROOT = REPO_ROOT.parent
sys.path.insert(0, str(REPO_ROOT))
print("Repo root:", REPO_ROOT)

# Install project deps not preinstalled on Kaggle.
# !pip install -q ultralytics

DATA_ROOT = None  # e.g. "/kaggle/input/smoke-fire-detection-yolo"
if DATA_ROOT:
    os.environ["DFIRE_ROOT"] = DATA_ROOT

CONFIG = str(REPO_ROOT / "configs" / "train_yolo_main.yaml")

In [ ]:
# Train (config-driven). Use a few epochs first as a smoke test, then full run.
from src.training.train_yolo import train

overrides = {"data_root": DATA_ROOT, "epochs": 5}  # set epochs=None to use the config value
run_dir = train(CONFIG, overrides)
best = Path(run_dir) / "weights" / "best.pt"
print("Best weights:", best)

In [ ]:
# Evaluate on the validation split (detection + alert metrics + calibration + tuned thresholds)
from src.evaluation.evaluate_detector import evaluate

report = evaluate(
    weights=str(best),
    split="val",
    data_root=DATA_ROOT,
    imgsz=640,
    iou=0.6,
    device_arg=None,
    inference_config=str(REPO_ROOT / "configs" / "inference.yaml"),
    do_tune=True,
)
print("mAP50-95:", report["detection"]["map50_95"], "| mAP50:", report["detection"]["map50"])
print("Per-class AP50-95:", report["detection"]["per_class_ap50_95"])
print("Tuned thresholds:", report["thresholds_tuned"])